# f4_m08_perfiles_riesgo.ipynb
## Fase 4 · EDA Final · Módulo 8 — Análisis Comparativo: Abandono vs Éxito


**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |

---

---

## Qué hace
Construye el **análisis comparativo estadístico y visual** entre el grupo de estudiantes
que abandona y el que completa la carrera, usando las features seleccionadas en M07.

Genera **dos HTMLs**:
- `m08_perfiles_riesgo.html` — análisis principal (radar, violín, KDE, barras, tabla semáforo)
- `m08_perfiles_riesgo_ampliacion.html` — análisis extendido (heatmap, beeswarm, sankey, bump, KDE por feature)

## Requisitos
- `data/automl/dataset_final_tfm.parquet`
- `data/03_features/f4_m06_correlaciones_tabla.parquet`
- `data/03_features/f4_m07_listas_features.json`

## Genera
- `docs/html/fase4/m08_perfiles_riesgo.html`
- `docs/html/fase4/m08_perfiles_riesgo_ampliacion.html`

## Flujo
`M07 Selección` → **M08 Comparativa** → `M09 Conclusiones`

## Siguiente
`f4_m09_conclusiones.ipynb`


In [1]:
# ============================================================================
# CELDA 1: IMPORTS Y CONFIGURACIÓN
# ============================================================================
import sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import json

import numpy  as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express       as px
from plotly.subplots import make_subplots

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists(): break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import RUTA_AUTOML, RUTA_FEATURES, RUTA_HTML, info_entorno
from src.utils  import crear_directorios, formato_numero_es
from src.html   import generar_html_navegacion_completa, guardar_html
from src.html.render import render_base_html

RUTA_FASE4_HTML = RUTA_HTML / 'fase4'
crear_directorios([RUTA_FASE4_HTML])

TARGET        = 'abandono'
COLOR_ABND    = '#e53e3e'
COLOR_EXIT    = '#3182ce'
N_TOP_RADAR   = 8
N_TOP_VIOLIN  = 6   # features en violin/KDE (menos = más legible)

TITULO_PAGINA = 'Análisis Comparativo: Abandono vs Éxito'
SUBTITULO     = 'Fase 4 · EDA Final · Módulo 8'

info_entorno()
print('Imports OK')


✓ Directorios verificados: 1
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

---
### 📖 Datos y división en grupos

Este módulo divide los 33.621 estudiantes en dos grupos según el target `abandono`
y calcula estadísticas comparativas. Todas las visualizaciones operan sobre estos
dos grupos de forma consistente: 🔴 rojo = abandonador, 🔵 azul = éxito/activo.


In [2]:
# ============================================================================
# CELDA 2: CARGA Y DIVISIÓN EN GRUPOS
# ============================================================================
RUTA_DS = RUTA_AUTOML / 'dataset_final_tfm.parquet'
assert RUTA_DS.exists(), f'No encontrado: {RUTA_DS}'
df = pd.read_parquet(RUTA_DS)
print(f'Dataset: {df.shape[0]:,} × {df.shape[1]}')

# Ranking M06
RUTA_RANKING = RUTA_FEATURES / 'f4_m06_correlaciones_tabla.parquet'
if RUTA_RANKING.exists():
    df_ranking = pd.read_parquet(RUTA_RANKING)
    df_ranking.columns = [c.lower().strip() for c in df_ranking.columns]
    TOP_FEATURES  = df_ranking.head(N_TOP_RADAR)['feature'].tolist()
    TOP_VIOLIN    = df_ranking.head(N_TOP_VIOLIN)['feature'].tolist()
    ALL_FEATURES  = df_ranking['feature'].tolist()
else:
    ALL_FEATURES = [c for c in df.columns if c != TARGET
                    and df[c].dtype in ['float64','int64']]
    TOP_FEATURES = ALL_FEATURES[:N_TOP_RADAR]
    TOP_VIOLIN   = ALL_FEATURES[:N_TOP_VIOLIN]

# Lista M07
RUTA_JSON = RUTA_FEATURES / 'f4_m07_listas_features.json'
if RUTA_JSON.exists():
    listas_m07 = json.loads(RUTA_JSON.read_text())
    FEATURES_CAT = [f for f in listas_m07.get('features_fase5_todas', ALL_FEATURES)
                    if df[f].dtype == 'object' or df[f].nunique() <= 10]
    FEATURES_NUM = [f for f in listas_m07.get('features_fase5_todas', ALL_FEATURES)
                    if f not in FEATURES_CAT]
else:
    FEATURES_CAT = [c for c in df.columns if c != TARGET and df[c].nunique() <= 10]
    FEATURES_NUM = [c for c in df.columns if c != TARGET and c not in FEATURES_CAT]

df_abnd = df[df[TARGET] == 1].copy()
df_exit = df[df[TARGET] == 0].copy()
n_abnd  = len(df_abnd)
n_exit  = len(df_exit)

print(f'Abandonadores : {n_abnd:,} ({n_abnd/len(df)*100:.1f}%)')
print(f'Éxito/Activos : {n_exit:,} ({n_exit/len(df)*100:.1f}%)')
print(f'Top features radar  : {TOP_FEATURES}')
print(f'Features categóricas: {FEATURES_CAT}')


Dataset: 33,621 × 20
Abandonadores : 9,833 (29.2%)
Éxito/Activos : 23,788 (70.8%)
Top features radar  : ['n_anios_beca', 'nota_1er_anio', 'cred_superados_anio_1er', 'nota_acceso', 'tuvo_beca', 'nota_selectividad', 'situacion_laboral', 'anios_sin_beca']
Features categóricas: ['n_anios_beca', 'tuvo_beca', 'via_acceso', 'indicador_interrupcion', 'pais_nombre', 'sexo', 'rama', 'universidad_origen', 'max_pagos', 'provincia', 'anios_gap']


In [3]:
# ============================================================================
# CELDA 3: ESTADÍSTICAS POR GRUPO
# ============================================================================
stats = []
for feat in ALL_FEATURES:
    if feat not in df.columns: continue
    if df[feat].dtype not in ['float64','int64','float32','int32']: continue
    med_a = df_abnd[feat].median()
    med_e = df_exit[feat].median()
    p25_a = df_abnd[feat].quantile(0.25)
    p75_a = df_abnd[feat].quantile(0.75)
    p25_e = df_exit[feat].quantile(0.25)
    p75_e = df_exit[feat].quantile(0.75)
    diff  = med_a - med_e
    stats.append({'feature': feat,
                  'med_abnd': med_a, 'p25_abnd': p25_a, 'p75_abnd': p75_a,
                  'med_exit': med_e, 'p25_exit': p25_e, 'p75_exit': p75_e,
                  'diff_abs': diff})

df_stats = pd.DataFrame(stats)
df_stats['diff_ord'] = df_stats['diff_abs'].abs()
df_stats = df_stats.sort_values('diff_ord', ascending=False).reset_index(drop=True)
print(df_stats[['feature','med_abnd','med_exit','diff_abs']].to_string(index=False))


                feature  med_abnd  med_exit  diff_abs
cred_superados_anio_1er     30.00     60.00    -30.00
      situacion_laboral     11.00      7.00      4.00
            nota_acceso      7.34      8.60     -1.26
           n_anios_beca      1.00      2.00     -1.00
                   sexo      1.00      0.00      1.00
           edad_entrada     19.00     18.00      1.00
          nota_1er_anio      6.23      6.96     -0.73
      nota_selectividad      6.18      6.75     -0.57
         anios_sin_beca      1.00      1.00      0.00
              tuvo_beca      1.00      1.00      0.00
                   rama      3.00      3.00      0.00
             via_acceso     10.00     10.00      0.00
     universidad_origen     40.00     40.00      0.00
              max_pagos      2.00      2.00      0.00
              provincia      1.00      1.00      0.00
            pais_nombre      1.00      1.00      0.00
              anios_gap      0.00      0.00      0.00
 indicador_interrupcion     

---
### 📖 Gráficos del análisis principal

Los cuatro gráficos del HTML principal se complementan:
- **Radar** — visión global de las top features, normalizado
- **Violín** — distribución completa de cada feature, no solo la mediana
- **KDE** — curvas de densidad superpuestas, fácil de leer para el tribunal
- **Barras apiladas** — proporciones reales para variables categóricas


In [4]:
# ============================================================================
# CELDA 4: RADAR + VIOLÍN + KDE + BARRAS CATEGÓRICAS
# ============================================================================

# ── Radar ────────────────────────────────────────────────────────────────────
feats_radar = [f for f in TOP_FEATURES if f in df_stats['feature'].values]
df_radar = df_stats[df_stats['feature'].isin(feats_radar)].copy()

for feat in feats_radar:
    col = df[feat].dropna()
    vmin, vmax = col.min(), col.max()
    rng = vmax - vmin if vmax != vmin else 1
    mask = df_radar['feature'] == feat
    df_radar.loc[mask, 'norm_abnd'] = ((df_radar.loc[mask, 'med_abnd'] - vmin) / rng).values
    df_radar.loc[mask, 'norm_exit'] = ((df_radar.loc[mask, 'med_exit'] - vmin) / rng).values

df_radar = df_radar.set_index('feature').loc[feats_radar].reset_index()
labels_r  = [f.replace('_',' ') for f in df_radar['feature'].tolist()]
vals_abnd = df_radar['norm_abnd'].tolist() + [df_radar['norm_abnd'].iloc[0]]
vals_exit = df_radar['norm_exit'].tolist() + [df_radar['norm_exit'].iloc[0]]
labs_cierre = labels_r + [labels_r[0]]

fig_radar = go.Figure()
fig_radar.add_trace(go.Scatterpolar(
    r=vals_abnd, theta=labs_cierre, fill='toself', name='Abandonador',
    line=dict(color=COLOR_ABND, width=2), fillcolor='rgba(229,62,62,0.15)',
    hovertemplate='<b>%{theta}</b><br>Norm.: %{r:.2f}<extra>Abandonador</extra>'
))
fig_radar.add_trace(go.Scatterpolar(
    r=vals_exit, theta=labs_cierre, fill='toself', name='Éxito / activo',
    line=dict(color=COLOR_EXIT, width=2), fillcolor='rgba(49,130,206,0.15)',
    hovertemplate='<b>%{theta}</b><br>Norm.: %{r:.2f}<extra>Éxito</extra>'
))
fig_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0,1],
               tickfont=dict(size=9), gridcolor='#e2e8f0'),
               angularaxis=dict(tickfont=dict(size=11))),
    showlegend=True,
    legend=dict(orientation='h', yanchor='top', y=-0.08, xanchor='center', x=0.5),
    title=dict(text=f'Perfil radar — top {len(feats_radar)} features (0–1)',
               font=dict(size=13), x=0.5),
    height=500, paper_bgcolor='white', margin=dict(t=60,b=80,l=60,r=60)
)
IMG_RADAR = fig_radar.to_html(full_html=False, include_plotlyjs='cdn')
print('✅ Radar OK')

# ── Split Violin — subplots por feature, escala P2–P98 ──────────────────────
feats_vln = [f for f in TOP_VIOLIN if f in df.columns
             and df[f].dtype in ['float64','int64','float32','int32']]

n_cols_v = 3
n_rows_v = (len(feats_vln) + n_cols_v - 1) // n_cols_v

fig_vln = make_subplots(
    rows=n_rows_v, cols=n_cols_v,
    subplot_titles=[f.replace('_',' ') for f in feats_vln],
    vertical_spacing=0.14, horizontal_spacing=0.08
)

for idx, feat in enumerate(feats_vln):
    r = idx // n_cols_v + 1
    c = idx % n_cols_v + 1
    vals_a = df_abnd[feat].dropna().values
    vals_e = df_exit[feat].dropna().values
    # Limitar eje Y al rango P1–P99 para evitar outliers extremos
    all_v  = np.concatenate([vals_a, vals_e])
    y_min  = np.percentile(all_v, 1)
    y_max  = np.percentile(all_v, 99)
    # Clip valores fuera de rango
    vals_a_c = np.clip(vals_a, y_min, y_max)
    vals_e_c = np.clip(vals_e, y_min, y_max)

    fig_vln.add_trace(go.Violin(
        y=vals_a_c, x0=feat.replace('_',' '),
        name='Abandonador', legendgroup='abnd',
        showlegend=(idx == 0), side='negative',
        line_color=COLOR_ABND, fillcolor='rgba(229,62,62,0.3)',
        box_visible=True, meanline_visible=True, points=False,
        hovertemplate=f'<b>{feat}</b><br>%{{y:.2f}}<extra>Abandonador</extra>'
    ), row=r, col=c)
    fig_vln.add_trace(go.Violin(
        y=vals_e_c, x0=feat.replace('_',' '),
        name='Éxito / Activo', legendgroup='exit',
        showlegend=(idx == 0), side='positive',
        line_color=COLOR_EXIT, fillcolor='rgba(49,130,206,0.3)',
        box_visible=True, meanline_visible=True, points=False,
        hovertemplate=f'<b>{feat}</b><br>%{{y:.2f}}<extra>Éxito / Activo</extra>'
    ), row=r, col=c)

    # Forzar eje Y al rango P1–P99
    axis_key = f'yaxis{idx+1}' if idx > 0 else 'yaxis'
    fig_vln.update_layout(**{
        axis_key: dict(range=[y_min - (y_max-y_min)*0.05,
                               y_max + (y_max-y_min)*0.05])
    })

fig_vln.update_layout(
    violingap=0, violinmode='overlay',
    height=max(420, n_rows_v * 310),
    legend=dict(orientation='h', yanchor='bottom', y=1.03,
                xanchor='center', x=0.5,
                bgcolor='rgba(255,255,255,0.9)',
                bordercolor='#e2e8f0', borderwidth=1),
    title=dict(
        text='Violín dividido — izquierda=Abandono · derecha=Éxito (escala P1–P99 por feature)',
        font=dict(size=12), x=0.5),
    paper_bgcolor='white', plot_bgcolor='white',
    margin=dict(t=80, b=80)
)
fig_vln.update_xaxes(showticklabels=False)
IMG_VIOLIN = fig_vln.to_html(full_html=False, include_plotlyjs='cdn')
print('✅ Split violin OK')

# ── Dot plot IC 95% — normalizado para comparar en mismo eje ─────────────────
from scipy import stats as scipy_stats

# Normalizar medias a 0-1 por feature para que cred_superados no rompa el eje
ic_rows = []
for feat in feats_vln:
    vals_a = df_abnd[feat].dropna().values
    vals_e = df_exit[feat].dropna().values
    all_v  = np.concatenate([vals_a, vals_e])
    vmin, vmax = all_v.min(), all_v.max()
    rng = vmax - vmin if vmax != vmin else 1.0

    for grp, vals, color in [
        ('Abandonador',    vals_a, COLOR_ABND),
        ('Éxito / Activo', vals_e, COLOR_EXIT)
    ]:
        n    = len(vals)
        mean = vals.mean()
        sem  = scipy_stats.sem(vals)
        ci   = scipy_stats.t.ppf(0.975, df=n-1) * sem
        # Normalizar a 0-1
        mean_n = (mean - vmin) / rng
        ci_n   = ci / rng
        ic_rows.append({
            'feature': feat.replace('_',' '),
            'grupo': grp, 'mean_n': mean_n, 'ci_n': ci_n, 'color': color
        })

import pandas as pd
df_ic = pd.DataFrame(ic_rows)

fig_dot = go.Figure()
for grp, color, symbol in [
    ('Abandonador',    COLOR_ABND, 'circle'),
    ('Éxito / Activo', COLOR_EXIT, 'diamond')
]:
    sub = df_ic[df_ic['grupo'] == grp]
    fig_dot.add_trace(go.Scatter(
        x=sub['mean_n'],
        y=sub['feature'],
        mode='markers',
        name=grp,
        marker=dict(color=color, size=11, symbol=symbol,
                    line=dict(color='white', width=1.5)),
        error_x=dict(type='data', array=sub['ci_n'].tolist(),
                     color=color, thickness=2.5, width=8),
        hovertemplate=(
            '<b>%{y}</b><br>'
            'Media norm.: %{x:.3f}<br>'
            'IC 95% visible en barra'
            '<extra>' + grp + '</extra>'
        )
    ))

fig_dot.update_layout(
    height=max(320, len(feats_vln) * 52),
    legend=dict(orientation='h', yanchor='bottom', y=1.04,
                xanchor='center', x=0.5),
    title=dict(
        text='Medias normalizadas (0–1) con IC 95% — barras sin tocarse = diferencia significativa',
        font=dict(size=12), x=0.5),
    paper_bgcolor='white', plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='#e2e8f0',
               title='Valor normalizado (0=mín · 1=máx de cada feature)',
               range=[-0.1, 1.15], tickformat='.2f'),
    yaxis=dict(autorange='reversed', tickfont=dict(size=11)),
    margin=dict(t=60, b=90, l=160, r=40)
)
IMG_DOT_IC = fig_dot.to_html(full_html=False, include_plotlyjs='cdn')
print('✅ Dot plot IC normalizado OK')

# ── KDE ──────────────────────────────────────────────────────────────────────
from scipy.stats import gaussian_kde

n_cols = 3
n_rows = (len(feats_vln) + n_cols - 1) // n_cols

fig_kde = make_subplots(rows=n_rows, cols=n_cols,
    subplot_titles=[f.replace('_',' ') for f in feats_vln])

for idx, feat in enumerate(feats_vln):
    r = idx // n_cols + 1
    c = idx % n_cols + 1
    for grp, color, df_g in [
        ('Abandonador',   COLOR_ABND, df_abnd),
        ('Éxito / Activo', COLOR_EXIT, df_exit)
    ]:
        vals = df_g[feat].dropna().values
        if len(vals) < 10: continue
        kde  = gaussian_kde(vals, bw_method='scott')
        x_rng = np.linspace(vals.min(), vals.max(), 200)
        y_kde = kde(x_rng)
        r_int = int(color.lstrip('#')[0:2], 16)
        g_int = int(color.lstrip('#')[2:4], 16)
        b_int = int(color.lstrip('#')[4:6], 16)
        fig_kde.add_trace(go.Scatter(
            x=x_rng, y=y_kde, mode='lines', name=grp,
            legendgroup=grp, showlegend=(idx == 0),
            line=dict(color=color, width=2),
            fill='tozeroy',
            fillcolor=f'rgba({r_int},{g_int},{b_int},0.15)',
            hovertemplate=f'<b>{feat}</b><br>x=%{{x:.2f}}<br>densidad=%{{y:.3f}}<extra>{grp}</extra>'
        ), row=r, col=c)

fig_kde.update_layout(
    height=max(350, n_rows * 280),
    legend=dict(orientation='h', yanchor='top', y=-0.05, xanchor='center', x=0.5),
    title=dict(text='Densidad de kernel (KDE) — distribución suavizada por grupo',
               font=dict(size=13), x=0.5),
    paper_bgcolor='white', plot_bgcolor='white',
    margin=dict(t=60, b=60)
)
IMG_KDE = fig_kde.to_html(full_html=False, include_plotlyjs='cdn')
print('✅ KDE OK')

# ── Barras apiladas — categóricas ────────────────────────────────────────────
feats_cat_plot = [f for f in FEATURES_CAT if f in df.columns][:6]

if feats_cat_plot:
    n_c2  = min(3, len(feats_cat_plot))
    n_r2  = (len(feats_cat_plot) + n_c2 - 1) // n_c2
    fig_cat = make_subplots(rows=n_r2, cols=n_c2,
        subplot_titles=[f.replace('_',' ') for f in feats_cat_plot])

    for idx, feat in enumerate(feats_cat_plot):
        r = idx // n_c2 + 1
        c = idx % n_c2 + 1
        cats = df[feat].dropna().unique()
        cats = sorted(cats, key=lambda x: str(x))[:10]
        pcts_abnd, pcts_exit = [], []
        for cat in cats:
            n_a = (df_abnd[feat] == cat).sum()
            n_e = (df_exit[feat] == cat).sum()
            tot = n_a + n_e
            pcts_abnd.append(round(n_a / tot * 100, 1) if tot > 0 else 0)
            pcts_exit.append(round(n_e / tot * 100, 1) if tot > 0 else 0)
        cats_str = [str(x)[:12] for x in cats]
        fig_cat.add_trace(go.Bar(
            name='Abandonador', x=cats_str, y=pcts_abnd,
            marker_color=COLOR_ABND, legendgroup='abnd',
            showlegend=(idx == 0), opacity=0.85,
            hovertemplate='<b>%{x}</b><br>Abandono: %{y:.1f}%<extra></extra>'
        ), row=r, col=c)
        fig_cat.add_trace(go.Bar(
            name='Éxito / Activo', x=cats_str, y=pcts_exit,
            marker_color=COLOR_EXIT, legendgroup='exit',
            showlegend=(idx == 0), opacity=0.85,
            hovertemplate='<b>%{x}</b><br>Éxito: %{y:.1f}%<extra></extra>'
        ), row=r, col=c)

    fig_cat.update_layout(
        barmode='group', height=max(350, n_r2 * 300),
        legend=dict(orientation='h', yanchor='top', y=-0.05, xanchor='center', x=0.5),
        title=dict(text='Variables categóricas — % abandono vs éxito por categoría',
                   font=dict(size=13), x=0.5),
        paper_bgcolor='white', plot_bgcolor='white',
        margin=dict(t=60, b=60)
    )
    IMG_CAT = fig_cat.to_html(full_html=False, include_plotlyjs='cdn')
    print('✅ Barras categóricas OK')
else:
    IMG_CAT = '<p style="color:#718096;font-style:italic;">No se detectaron variables categóricas.</p>'
    print('⚠️  Sin features categóricas')


✅ Radar OK


✅ Split violin OK


✅ Dot plot IC normalizado OK


✅ KDE OK
✅ Barras categóricas OK


## 📐 Criterio del semáforo — d de Cohen

La **tabla semáforo** usa la **d de Cohen** para clasificar cada feature.

### ¿Qué es la d de Cohen?
Mide cuántas desviaciones típicas separan las medias de dos grupos.  
Fórmula: `d = |media_abandono - media_éxito| / desviación_típica_pooled`

### Umbrales universalmente aceptados (Cohen, 1988)
| d | Interpretación |
|---|---|
| < 0.2 | 🟡 Efecto negligible — los grupos son prácticamente iguales |
| 0.2 – 0.5 | 🟡 Efecto pequeño — diferencia existe pero es débil |
| 0.5 – 0.8 | 🟢🔴 Efecto medio — diferencia clara y prácticamente relevante |
| ≥ 0.8 | 🟢🔴 Efecto grande — diferencia muy marcada |

### Lógica del semáforo
- 🟢 **Protector**: d ≥ 0.5 **y** media éxito > media abandono  
- 🔴 **Factor de riesgo**: d ≥ 0.5 **y** media abandono > media éxito  
- 🟡 **Solapamiento**: d < 0.5 (diferencia existe pero es débil)

### ¿Por qué no P25–P75?
Con 33.621 estudiantes y variables como notas (rango 5–10), los rangos  
intercuartílicos **siempre se solapan** aunque las medianas difieran claramente.  
El criterio P25–P75 es demasiado estricto para datos educativos reales.

### Influencia en Fase 5
Las features 🟢 y 🔴 (d ≥ 0.5) son **candidatas prioritarias** para:
- Selección de features en modelos clásicos
- Justificación de hiperparámetros en Optuna
- Análisis SHAP en Fase 6 (interpretabilidad)

### Referencia
Cohen, J. (1988). *Statistical Power Analysis for the Behavioral Sciences* (2nd ed.).  
Lawrence Erlbaum Associates. ISBN 0-8058-0283-5


In [5]:
# ============================================================================
# CELDA 5: COMPARATIVA BARRAS + TABLA SEMÁFORO
# ============================================================================

# ── Comparativa barras medianas ───────────────────────────────────────────────
n_comp  = min(12, len(df_stats))
df_comp = df_stats.head(n_comp).copy()
labs_comp = [f.replace('_',' ') for f in df_comp['feature']]

fig_comp = make_subplots(rows=1, cols=2,
    subplot_titles=('Mediana — Abandonador', 'Mediana — Éxito / Activo'),
    horizontal_spacing=0.08)

fig_comp.add_trace(go.Bar(
    name='Abandonador', y=labs_comp, x=df_comp['med_abnd'],
    orientation='h', marker_color=COLOR_ABND, opacity=0.85,
    error_x=dict(type='data', symmetric=False,
                 array=(df_comp['p75_abnd']-df_comp['med_abnd']).tolist(),
                 arrayminus=(df_comp['med_abnd']-df_comp['p25_abnd']).tolist(),
                 color='#718096', thickness=1.5, width=4),
    hovertemplate='<b>%{y}</b><br>Mediana: %{x:.2f}<extra>Abandonador</extra>'
), row=1, col=1)

fig_comp.add_trace(go.Bar(
    name='Éxito / Activo', y=labs_comp, x=df_comp['med_exit'],
    orientation='h', marker_color=COLOR_EXIT, opacity=0.85,
    error_x=dict(type='data', symmetric=False,
                 array=(df_comp['p75_exit']-df_comp['med_exit']).tolist(),
                 arrayminus=(df_comp['med_exit']-df_comp['p25_exit']).tolist(),
                 color='#718096', thickness=1.5, width=4),
    hovertemplate='<b>%{y}</b><br>Mediana: %{x:.2f}<extra>Éxito</extra>'
), row=1, col=2)

fig_comp.update_layout(
    height=max(400, n_comp*32), showlegend=False,
    plot_bgcolor='white', paper_bgcolor='white',
    margin=dict(t=50,b=40,l=10,r=10), font=dict(size=11)
)
fig_comp.update_xaxes(showgrid=True, gridcolor='#e2e8f0')
fig_comp.update_yaxes(autorange='reversed')
IMG_COMP = fig_comp.to_html(full_html=False, include_plotlyjs='cdn')
print('✅ Comparativa barras OK')

# ── Tabla semáforo — criterio d de Cohen (Cohen, 1988) ───────────────────────
# 🟢 Protector:       d ≥ 0.5 y éxito > abandono
# 🔴 Factor riesgo:   d ≥ 0.5 y abandono > éxito
# 🟡 Solapamiento:    d < 0.5 (diferencia débil independientemente de dirección)

def cohens_d_sem(a, b):
    sp = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
    return abs(np.mean(a) - np.mean(b)) / sp if sp > 0 else 0

filas_sem = []
for _, row in df_stats.head(15).iterrows():
    feat  = row['feature']
    a_vals = df_abnd[feat].dropna().values
    e_vals = df_exit[feat].dropna().values
    d      = cohens_d_sem(a_vals, e_vals)
    med_a  = row['med_abnd']
    med_e  = row['med_exit']
    diff   = med_e - med_a   # positivo = éxito mayor

    if d >= 0.8:
        mag = 'Grande'
    elif d >= 0.5:
        mag = 'Medio'
    elif d >= 0.2:
        mag = 'Pequeño'
    else:
        mag = 'Negligible'

    if d >= 0.5 and diff > 0:
        icono = '🟢'
        etiq  = 'Protector'
        color_fila = 'background:#f0fff4;'
    elif d >= 0.5 and diff < 0:
        icono = '🔴'
        etiq  = 'Factor de riesgo'
        color_fila = 'background:#fff5f5;'
    else:
        icono = '🟡'
        etiq  = 'Solapamiento'
        color_fila = 'background:#fffff0;'

    filas_sem.append({
        'feature': feat, 'icono': icono, 'etiq': etiq,
        'med_a': med_a, 'med_e': med_e, 'diff': diff,
        'd': d, 'mag': mag, 'color_fila': color_fila
    })

# Ordenar: primero 🔴, luego 🟢, luego 🟡, dentro de cada grupo por d desc
orden = {'🔴': 0, '🟢': 1, '🟡': 2}
filas_sem.sort(key=lambda x: (orden[x['icono']], -x['d']))

filas_html = ''
for f in filas_sem:
    flecha = '↑ éxito' if f['diff'] > 0 else '↓ abandono'
    filas_html += (
        f'<tr style="{f["color_fila"]}">'
        f'<td style="padding:8px 12px;font-size:13px;">{f["icono"]} {f["etiq"]}</td>'
        f'<td style="padding:8px 12px;font-size:13px;font-weight:600;">'
        f'{f["feature"].replace("_"," ")}</td>'
        f'<td style="padding:8px 12px;font-size:13px;text-align:center;">'
        f'{f["med_a"]:.2f}</td>'
        f'<td style="padding:8px 12px;font-size:13px;text-align:center;">'
        f'{f["med_e"]:.2f}</td>'
        f'<td style="padding:8px 12px;font-size:13px;text-align:center;font-weight:600;">'
        f'{abs(f["diff"]):.2f} {flecha}</td>'
        f'<td style="padding:8px 12px;font-size:13px;text-align:center;">'
        f'{f["d"]:.2f}</td>'
        f'<td style="padding:8px 12px;font-size:13px;text-align:center;">'
        f'{f["mag"]}</td>'
        f'</tr>'
    )

tabla_semaforo_html = (
    '<div style="overflow-x:auto;">'
    '<table style="width:100%;border-collapse:collapse;font-size:13px;">'
    '<thead>'
    '<tr style="background:#2d3748;color:#fff;">'
    '<th style="padding:10px 12px;text-align:left;">Estado</th>'
    '<th style="padding:10px 12px;text-align:left;">Feature</th>'
    '<th style="padding:10px 12px;text-align:center;">Med. abandono</th>'
    '<th style="padding:10px 12px;text-align:center;">Med. éxito</th>'
    '<th style="padding:10px 12px;text-align:center;">Diferencia</th>'
    '<th style="padding:10px 12px;text-align:center;">d Cohen</th>'
    '<th style="padding:10px 12px;text-align:center;">Magnitud</th>'
    '</tr>'
    '</thead>'
    f'<tbody>{filas_html}</tbody>'
    '</table>'
    '</div>'
    '<p style="color:#718096;font-size:11px;margin-top:8px;">'
    '🟢 Protector (d≥0.5, éxito mayor) · '
    '🔴 Factor de riesgo (d≥0.5, abandono mayor) · '
    '🟡 Solapamiento (d&lt;0.5) · '
    'Referencia: Cohen (1988)</p>'
)
print('✅ Tabla semáforo con d de Cohen OK')

# ============================================================================

# ============================================================================
# INTERPRETACIONES DINÁMICAS — generadas desde los datos
# ============================================================================

def cohens_d(a, b):
    sp = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
    return abs(np.mean(a) - np.mean(b)) / sp if sp > 0 else 0

from scipy.stats import sem as scipy_sem, t as t_dist
from scipy.stats import kurtosis, skew

# ── KDE ──────────────────────────────────────────────────────────────────────
kde_items = []
kde_d_vals = []
for feat in feats_vln:
    a = df_abnd[feat].dropna().values
    e = df_exit[feat].dropna().values
    if len(a) < 10 or len(e) < 10: continue
    d = cohens_d(a, e)
    kde_d_vals.append((feat, d))
    dir_txt = 'abandono &gt; éxito' if a.mean() > e.mean() else 'éxito &gt; abandono'
    if d >= 0.8:   tag = '<span class="tg">✅ Muy buen predictor</span>'
    elif d >= 0.5: tag = '<span class="tg">✅ Buen predictor</span>'
    elif d >= 0.2: tag = '<span class="tm">⚠️ Discrimina moderadamente</span>'
    else:          tag = '<span class="tb">❌ Discrimina poco</span>'
    kde_items.append((feat, d,
        f'<li><b>{feat.replace("_"," ")}</b> — media abandono={a.mean():.2f}, '
        f'éxito={e.mean():.2f} ({dir_txt}). d={d:.2f}. {tag}</li>'
    ))
kde_items_sorted = sorted(kde_items, key=lambda x: -x[1])
best_kde  = kde_items_sorted[0][0].replace('_',' ') if kde_items_sorted else '—'
worst_kde = kde_items_sorted[-1][0].replace('_',' ') if kde_items_sorted else '—'
best_d    = kde_items_sorted[0][1] if kde_items_sorted else 0
worst_d   = kde_items_sorted[-1][1] if kde_items_sorted else 0
kde_interp_html = (
    '<b>Interpretación por feature</b> '
    '<span style="font-size:11px;color:#718096;">'
    '(d de Cohen: &lt;0.2 no discrimina · 0.2–0.5 moderado · 0.5–0.8 bueno · &gt;0.8 muy bueno)</span>'
    f'<ul style="margin-top:8px;">{"".join(x[2] for x in kde_items_sorted)}</ul>'
    f'<div style="margin-top:8px;padding:8px 12px;background:#fff;border-left:3px solid #3182ce;'
    f'border-radius:4px;font-size:12px;color:#2d3748;">'
    f'🏆 <b>Mejor predictor:</b> {best_kde} (d={best_d:.2f}) · '
    f'⚠️ <b>Menor discriminación:</b> {worst_kde} (d={worst_d:.2f})'
    f'</div>'
)

# ── Ridgeline ────────────────────────────────────────────────────────────────
ridge_feats = [f for f in ALL_FEATURES if f in df.columns
               and df[f].dtype in ['float64','int64','float32','int32']
               and df[f].nunique() > 15][:8]
n_total_num = sum(1 for f in ALL_FEATURES if f in df.columns
                  and df[f].dtype in ['float64','int64','float32','int32'])
n_continuas = len(ridge_feats)
ridge_items = []
for feat in ridge_feats:
    a = df_abnd[feat].dropna().values
    e = df_exit[feat].dropna().values
    if len(a) < 10 or len(e) < 10: continue
    d = cohens_d(a, e)
    if d >= 0.5:   ico, desc = '🟢', 'curvas bien separadas'
    elif d >= 0.2: ico, desc = '🟡', 'solapamiento moderado'
    else:          ico, desc = '🔴', 'curvas muy solapadas'
    dir_txt = 'abandono concentrado antes' if a.mean() < e.mean() else 'éxito concentrado antes'
    ridge_items.append(
        f'<li>{ico} <b>{feat.replace("_"," ")}</b> — {desc} (d={d:.2f}). {dir_txt}.</li>'
    )
n_excluidas = n_total_num - n_continuas
ridge_interp_html = (
    f'<b>Separación entre grupos — {n_continuas} variables continuas</b><br>'
    f'<span style="font-size:11px;color:#718096;">'
    f'Filtro aplicado: solo features con &gt;15 valores únicos. '
    f'{n_excluidas} variable(s) discreta(s) excluida(s) del ridgeline '
    f'(aparecen en KDE y violín con escala independiente). '
    f'🟢 bien separado · 🟡 moderado · 🔴 muy solapado</span>'
    f'<ul style="margin-top:8px;">{"".join(ridge_items)}</ul>'
)

# ── Radar ─────────────────────────────────────────────────────────────────────
radar_rows = []
for feat in feats_radar:
    row = df_stats[df_stats['feature'] == feat]
    if row.empty: continue
    med_a = float(row['med_abnd'].values[0])
    med_e = float(row['med_exit'].values[0])
    d     = cohens_d(df_abnd[feat].dropna().values, df_exit[feat].dropna().values)
    diff  = med_e - med_a
    ref   = max(abs(med_a), abs(med_e), 0.01)
    if abs(diff) / ref < 0.05:
        desc = f'<span class="tm">grupos similares (aban={med_a:.2f}, éxito={med_e:.2f})</span>'
    elif diff > 0:
        desc = f'<span class="tg">éxito domina (aban={med_a:.2f}, éxito={med_e:.2f}, +{diff:.2f})</span>'
    else:
        desc = f'<span class="tb">abandono domina (aban={med_a:.2f}, éxito={med_e:.2f}, {diff:.2f})</span>'
    mag = 'grande' if d >= 0.8 else 'medio' if d >= 0.5 else 'pequeño' if d >= 0.2 else 'negligible'
    radar_rows.append((feat, abs(diff), d,
        f'<li><b>{feat.replace("_"," ")}</b> — {desc}. d={d:.2f} ({mag})</li>'
    ))
radar_rows_sorted = sorted(radar_rows, key=lambda x: -x[1])
radar_max = radar_rows_sorted[0][0].replace('_',' ') if radar_rows_sorted else '—'
radar_min = radar_rows_sorted[-1][0].replace('_',' ') if radar_rows_sorted else '—'
radar_interp_html = (
    '<b>Dominancia por dimensión del radar</b><br>'
    '<span style="font-size:11px;color:#718096;">'
    '<span class="tg">Verde</span> = éxito mayor · '
    '<span class="tb">Rojo</span> = abandono mayor · '
    '<span class="tm">Naranja</span> = grupos similares</span>'
    f'<ul style="margin-top:8px;">{"".join(x[3] for x in radar_rows_sorted)}</ul>'
    f'<div style="margin-top:8px;padding:8px 12px;background:#fff;border-left:3px solid #3182ce;'
    f'border-radius:4px;font-size:12px;color:#2d3748;">'
    f'📐 <b>Mayor diferencia en el radar:</b> {radar_max} · '
    f'<b>Menor diferencia:</b> {radar_min}'
    f'</div>'
)

# ── Violín ────────────────────────────────────────────────────────────────────
violin_items = []
for feat in feats_vln:
    a = df_abnd[feat].dropna().values
    e = df_exit[feat].dropna().values
    if len(a) < 10 or len(e) < 10: continue
    med_a, med_e = np.median(a), np.median(e)
    p25_a, p75_a = np.percentile(a, 25), np.percentile(a, 75)
    p25_e, p75_e = np.percentile(e, 25), np.percentile(e, 75)
    solapa = not (p75_a < p25_e or p75_e < p25_a)
    d = cohens_d(a, e)
    # Forma del violín
    sk_a = skew(a)
    sk_e = skew(e)
    if abs(sk_a) < 0.5:   forma_a = 'simétrico'
    elif sk_a > 0:         forma_a = 'cola derecha'
    else:                  forma_a = 'cola izquierda'
    if abs(sk_e) < 0.5:   forma_e = 'simétrico'
    elif sk_e > 0:         forma_e = 'cola derecha'
    else:                  forma_e = 'cola izquierda'
    # Estado
    if d >= 0.5 and not solapa and med_a > med_e:
        sep = '<span class="tb">🔴 abandono mayor, rangos separados</span>'
    elif d >= 0.5 and not solapa and med_e > med_a:
        sep = '<span class="tg">🟢 éxito mayor, rangos separados</span>'
    elif d >= 0.5 and solapa:
        sep = f'<span class="tm">⚠️ diferencia media-grande (d={d:.2f}) pero rangos solapan</span>'
    else:
        sep = f'<span class="tm">🟡 diferencia débil (d={d:.2f})</span>'
    violin_items.append(
        f'<li><b>{feat.replace("_"," ")}</b> — mediana abandono={med_a:.2f} ({forma_a}), '
        f'éxito={med_e:.2f} ({forma_e}). {sep}</li>'
    )
violin_interp_html = (
    '<b>Medianas, forma de distribución y separación por feature</b><br>'
    '<span style="font-size:11px;color:#718096;">'
    'Izquierda roja = abandonadores · Derecha azul = éxito. '
    'Escala P1–P99 por subplot. '
    'Forma: simétrico / cola derecha / cola izquierda según asimetría.</span>'
    f'<ul style="margin-top:8px;">{"".join(violin_items)}</ul>'
)

# ── Dot IC ────────────────────────────────────────────────────────────────────
dot_items = []
n_sig = 0
for feat in feats_vln:
    a = df_abnd[feat].dropna().values
    e = df_exit[feat].dropna().values
    if len(a) < 10 or len(e) < 10: continue
    vmin = min(a.min(), e.min()); vmax = max(a.max(), e.max())
    rng  = vmax - vmin if vmax != vmin else 1.0
    mean_a_n = (a.mean() - vmin) / rng
    mean_e_n = (e.mean() - vmin) / rng
    ci_a = t_dist.ppf(0.975, df=len(a)-1) * scipy_sem(a) / rng
    ci_e = t_dist.ppf(0.975, df=len(e)-1) * scipy_sem(e) / rng
    solapan = not (mean_a_n - ci_a > mean_e_n + ci_e or
                   mean_e_n - ci_e > mean_a_n + ci_a)
    d = cohens_d(a, e)
    if not solapan:
        n_sig += 1
        sig = f'<span class="tg">✅ Significativa (d={d:.2f})</span>'
    else:
        sig = f'<span class="tb">⚠️ No significativa (d={d:.2f})</span>'
    dot_items.append(
        f'<li><b>{feat.replace("_"," ")}</b> — '
        f'media norm. abandono={mean_a_n:.3f}±{ci_a:.3f}, '
        f'éxito={mean_e_n:.3f}±{ci_e:.3f}. {sig}</li>'
    )
n_no_sig = len(dot_items) - n_sig
dot_interp_html = (
    '<b>Significación estadística por feature (IC 95%)</b><br>'
    '<span style="font-size:11px;color:#718096;">'
    'Valores normalizados 0–1. Barras sin tocarse = diferencia significativa al 95%.</span>'
    f'<ul style="margin-top:8px;">{"".join(dot_items)}</ul>'
    f'<div style="margin-top:8px;padding:8px 12px;background:#fff;border-left:3px solid #3182ce;'
    f'border-radius:4px;font-size:12px;color:#2d3748;">'
    f'📊 <b>Resumen:</b> {n_sig} de {len(dot_items)} features con diferencia '
    f'estadísticamente significativa al 95% · {n_no_sig} no significativa(s)'
    f'</div>'
)

# ── Barras categóricas ────────────────────────────────────────────────────────
cat_items = []
feats_cat_plot = [f for f in FEATURES_CAT if f in df.columns][:6]
for feat in feats_cat_plot:
    cats = df[feat].dropna().value_counts().head(6).index.tolist()
    best_abnd = max(cats, key=lambda c: (
        (df_abnd[feat] == c).sum() /
        max((df_abnd[feat] == c).sum() + (df_exit[feat] == c).sum(), 1)
    ))
    best_exit = max(cats, key=lambda c: (
        (df_exit[feat] == c).sum() /
        max((df_abnd[feat] == c).sum() + (df_exit[feat] == c).sum(), 1)
    ))
    # % abandono en categoría de mayor riesgo
    n_a_risk  = (df_abnd[feat] == best_abnd).sum()
    n_tot_risk = n_a_risk + (df_exit[feat] == best_abnd).sum()
    pct_abnd  = round(n_a_risk / n_tot_risk * 100, 1) if n_tot_risk > 0 else 0
    # % éxito en categoría de mayor éxito
    n_e_best  = (df_exit[feat] == best_exit).sum()
    n_tot_best = (df_abnd[feat] == best_exit).sum() + n_e_best
    pct_exit  = round(n_e_best / n_tot_best * 100, 1) if n_tot_best > 0 else 0
    cat_items.append(
        f'<li><b>{feat.replace("_"," ")}</b> — '
        f'mayor riesgo: <b>{str(best_abnd)[:15]}</b> '
        f'<span class="tb">({pct_abnd}% abandona)</span> · '
        f'mayor éxito: <b>{str(best_exit)[:15]}</b> '
        f'<span class="tg">({pct_exit}% tiene éxito)</span></li>'
    )
cat_interp_html = (
    '<b>Categorías con mayor riesgo y mayor éxito por variable</b><br>'
    '<span style="font-size:11px;color:#718096;">'
    'Rojo = % de esa categoría que abandona · Verde = % de esa categoría con éxito.</span>'
    f'<ul style="margin-top:8px;">'
    f'{"".join(cat_items) if cat_items else "<li>No hay variables categóricas disponibles.</li>"}'
    f'</ul>'
)

# ── Comparativa medianas ──────────────────────────────────────────────────────
comp_rows = []
for _, row in df_stats.head(12).iterrows():
    diff = row['diff_abs']
    d    = cohens_d(df_abnd[row['feature']].dropna().values,
                    df_exit[row['feature']].dropna().values)
    if diff > 0:
        dir_txt = f'<span class="tb">abandono mayor (+{diff:.2f})</span>'
    elif diff < 0:
        dir_txt = f'<span class="tg">éxito mayor ({diff:.2f})</span>'
    else:
        dir_txt = '<span class="tm">sin diferencia</span>'
    solapa = not (row['p75_abnd'] < row['p25_exit'] or row['p75_exit'] < row['p25_abnd'])
    sep_txt = ' · <span class="tm">P25–P75 solapan</span>' if solapa               else ' · <span class="tg">rangos separados</span>'
    comp_rows.append((row['feature'], abs(diff), d,
        f'<li><b>{row["feature"].replace("_"," ")}</b> — '
        f'abandono={row["med_abnd"]:.2f}, éxito={row["med_exit"]:.2f}. '
        f'{dir_txt}{sep_txt}. d={d:.2f}</li>'
    ))
top3_comp = ', '.join(r[0].replace('_',' ') for r in comp_rows[:3])
comp_interp_html = (
    '<b>Diferencia de medianas con d de Cohen</b><br>'
    f'<div style="margin-bottom:8px;padding:8px 12px;background:#fff;'
    f'border-left:3px solid #3182ce;border-radius:4px;font-size:12px;color:#2d3748;">'
    f'🏆 <b>Top 3 mayor diferencia:</b> {top3_comp}'
    f'</div>'
    '<span style="font-size:11px;color:#718096;">'
    'Ordenadas de mayor a menor diferencia absoluta.</span>'
    f'<ul style="margin-top:8px;">{"".join(r[3] for r in comp_rows)}</ul>'
)

# ── Semáforo ──────────────────────────────────────────────────────────────────
n_verde    = sum(1 for f in filas_sem if f['icono'] == '🟢')
n_rojo     = sum(1 for f in filas_sem if f['icono'] == '🔴')
n_amarillo = sum(1 for f in filas_sem if f['icono'] == '🟡')
feat_verde_top = [f['feature'] for f in filas_sem if f['icono'] == '🟢'][:3]
feat_rojo_top  = [f['feature'] for f in filas_sem if f['icono'] == '🔴'][:3]
sem_top_verde  = ', '.join(f.replace('_',' ') for f in feat_verde_top) or '—'
sem_top_rojo   = ', '.join(f.replace('_',' ') for f in feat_rojo_top) or '—'

ausencia_rojo = (
    f'<div style="margin-top:10px;padding:8px 12px;background:#fff;'
    f'border-left:3px solid #e53e3e;border-radius:4px;font-size:12px;color:#2d3748;">'
    f'🔍 <b>Nota metodológica:</b> la ausencia de features 🔴 no es un defecto del análisis — '
    f'es un hallazgo en sí mismo. Indica que el abandono no se asocia a la presencia de '
    f'factores negativos propios, sino al déficit de factores protectores. '
    f'Esto es coherente con la literatura sobre abandono universitario (Tinto, 1987).'
    f'</div>'
) if n_rojo == 0 else ''

sem_interp_html = (
    f'<b>Resumen — {len(filas_sem)} features (criterio: d de Cohen ≥ 0.5):</b>'
    f'<ul style="margin-top:8px;">'
    f'<li><span class="tg">🟢 Protectoras: {n_verde}</span> — éxito mayor con efecto medio/grande. '
    f'Principales: {sem_top_verde}</li>'
    f'<li><span class="tb">🔴 Factor de riesgo: {n_rojo}</span> — abandono mayor con efecto medio/grande. '
    f'Principales: {sem_top_rojo if sem_top_rojo != "—" else "ninguna supera d≥0.5"}</li>'
    f'<li><span class="tm">🟡 Solapamiento: {n_amarillo}</span> — d&lt;0.5, diferencia débil.</li>'
    f'</ul>'
    f'<b>Prioridad para Fase 5 y 6:</b> features 🟢 y 🔴 → selección de features, Optuna y SHAP.<br>'
    f'<span style="font-size:11px;color:#718096;">'
    f'Cohen (1988): d&lt;0.2 negligible · 0.2–0.5 pequeño · 0.5–0.8 medio · ≥0.8 grande</span>'
    + ausencia_rojo
    + f'<div style="margin-top:10px;padding:8px 12px;background:#fff;'
    f'border-left:3px solid #3182ce;border-radius:4px;font-size:12px;color:#2d3748;">'
    f'💡 <b>Interpretación del patrón:</b> '
    f'el abandono se explica por <b>déficit en recursos académicos y de apoyo</b> '
    f'(menos créditos superados, peor nota en primer año, menos beca), '
    f'no por la presencia de factores negativos propios. '
    f'Este patrón orienta la intervención hacia el <b>refuerzo temprano en primer curso</b>. '
    f'Las variables 🟢 son candidatas prioritarias para SHAP en Fase 6.'
    f'</div>'
)

print(f'✅ Todas las interpretaciones dinámicas OK')
print(f'   KDE mejor: {best_kde} (d={best_d:.2f}) · peor: {worst_kde} (d={worst_d:.2f})')
print(f'   Dot IC: {n_sig}/{len(dot_items)} significativas')
print(f'   Semáforo: {n_verde}🟢 {n_rojo}🔴 {n_amarillo}🟡')
print(f'   Ridgeline: {n_continuas} continuas de {n_total_num} numéricas ({n_excluidas} excluidas)')


✅ Comparativa barras OK
✅ Tabla semáforo con d de Cohen OK


✅ Todas las interpretaciones dinámicas OK
   KDE mejor: n anios beca (d=0.92) · peor: tuvo beca (d=0.52)
   Dot IC: 6/6 significativas
   Semáforo: 5🟢 0🔴 10🟡
   Ridgeline: 6 continuas de 19 numéricas (13 excluidas)


In [6]:
# ============================================================================
# CELDA 6: GRÁFICOS DE AMPLIACIÓN
# ============================================================================

# ── Heatmap correlación por grupo ────────────────────────────────────────────
import plotly.figure_factory as ff

feats_heat = [f for f in ALL_FEATURES if f in df.columns
              and df[f].dtype in ['float64','int64','float32','int32']][:12]

def heatmap_grupo(df_g, titulo, colorscale):
    corr = df_g[feats_heat].corr().round(2)
    labs = [f.replace('_',' ')[:10] for f in feats_heat]
    fig = go.Figure(go.Heatmap(
        z=corr.values, x=labs, y=labs,
        colorscale=colorscale, zmid=0, zmin=-1, zmax=1,
        text=corr.values.round(2),
        texttemplate='%{text}', textfont=dict(size=9),
        hovertemplate='%{y} × %{x}<br>r = %{z:.2f}<extra></extra>'
    ))
    fig.update_layout(
        title=dict(text=titulo, font=dict(size=12), x=0.5),
        height=500, width=520,
        paper_bgcolor='white',
        margin=dict(t=50,b=10,l=10,r=10),
        xaxis=dict(tickfont=dict(size=9)),
        yaxis=dict(tickfont=dict(size=9), autorange='reversed')
    )
    return fig.to_html(full_html=False, include_plotlyjs='cdn')

IMG_HEAT_ABND = heatmap_grupo(df_abnd, 'Correlaciones — Grupo Abandonador', 'RdBu_r')
IMG_HEAT_EXIT = heatmap_grupo(df_exit, 'Correlaciones — Grupo Éxito/Activo', 'Blues')
print('✅ Heatmaps OK')

# ── Ridgeline plot ───────────────────────────────────────────────────────────
from scipy.stats import gaussian_kde

# Solo variables continuas (nunique > 15) — las discretas van en KDE y violín
feats_ridge = [f for f in ALL_FEATURES if f in df.columns
               and df[f].dtype in ['float64','int64','float32','int32']
               and df[f].nunique() > 15][:8]

n_feats     = len(feats_ridge)
offset_step = 1.0    # espacio normalizado entre filas
curve_height = 0.7   # altura máxima de cada curva — nunca supera offset_step

fig_ridge = go.Figure()

for idx, feat in enumerate(reversed(feats_ridge)):
    offset = idx * offset_step
    all_v  = df[feat].dropna().values
    # Clip P2-P98 para eliminar outliers que estiran el eje
    p2, p98 = np.percentile(all_v, 2), np.percentile(all_v, 98)
    vmin, vmax = p2, p98
    rng = vmax - vmin if vmax != vmin else 1.0

    # Banda de fondo alternada
    if idx % 2 == 0:
        fig_ridge.add_shape(type='rect',
            x0=-0.02, x1=1.02,
            y0=offset, y1=offset + offset_step,
            fillcolor='rgba(237,242,247,0.5)', line_width=0, layer='below')

    for grp, df_g, color in [
        ('Abandonador',    df_abnd, COLOR_ABND),
        ('Éxito / Activo', df_exit, COLOR_EXIT)
    ]:
        vals   = df_g[feat].dropna().values
        vals_c = np.clip(vals, vmin, vmax)   # clip outliers
        if len(vals_c) < 10: continue
        vals_n = (vals_c - vmin) / rng       # normalizar 0-1
        kde    = gaussian_kde(vals_n, bw_method=0.12)
        x_rng  = np.linspace(0, 1, 300)
        y_raw  = kde(x_rng)
        # Escalar para que la curva nunca supere curve_height
        y_scaled = y_raw / y_raw.max() * curve_height + offset
        r_int = int(color.lstrip('#')[0:2], 16)
        g_int = int(color.lstrip('#')[2:4], 16)
        b_int = int(color.lstrip('#')[4:6], 16)
        # Baseline para fill correcto
        fig_ridge.add_trace(go.Scatter(
            x=x_rng, y=[offset] * len(x_rng),
            mode='lines', line=dict(width=0),
            showlegend=False, hoverinfo='skip'
        ))
        fig_ridge.add_trace(go.Scatter(
            x=x_rng, y=y_scaled,
            mode='lines', name=grp, legendgroup=grp,
            showlegend=(idx == 0),
            line=dict(color=color, width=2),
            fill='tonexty',
            fillcolor=f'rgba({r_int},{g_int},{b_int},0.25)',
            hovertemplate=f'<b>{feat}</b><br>Valor norm.: %{{x:.2f}}<extra>{grp}</extra>'
        ))

tickvals = [(idx + 0.35) * offset_step for idx in range(n_feats)]
ticktext  = [f.replace('_',' ') for f in reversed(feats_ridge)]

fig_ridge.update_layout(
    height=120 + n_feats * 90,
    xaxis=dict(
        title='Valor normalizado (0 = mínimo · 1 = máximo, P2–P98)',
        showgrid=True, gridcolor='#e2e8f0',
        range=[-0.02, 1.05], tickformat='.1f'
    ),
    yaxis=dict(
        tickvals=tickvals, ticktext=ticktext,
        tickfont=dict(size=11), showgrid=False,
        range=[-0.05, n_feats * offset_step + 0.05]
    ),
    legend=dict(
        orientation='h', yanchor='bottom', y=1.02,
        xanchor='center', x=0.5, font=dict(size=12),
        bgcolor='rgba(255,255,255,0.95)',
        bordercolor='#e2e8f0', borderwidth=1
    ),
    paper_bgcolor='white', plot_bgcolor='white',
    margin=dict(t=50, b=70, l=155, r=20),
)
IMG_RIDGE = fig_ridge.to_html(full_html=False, include_plotlyjs='cdn')
print('✅ Ridgeline OK')

# ── Sankey ───────────────────────────────────────────────────────────────────
# Flujo: vía de acceso → situación laboral → resultado
feat_s1 = 'via_acceso'    if 'via_acceso'     in df.columns else FEATURES_CAT[0] if FEATURES_CAT else None
feat_s2 = 'situacion_laboral' if 'situacion_laboral' in df.columns else FEATURES_CAT[1] if len(FEATURES_CAT)>1 else None

if feat_s1 and feat_s2:
    cats1 = df[feat_s1].dropna().value_counts().head(4).index.tolist()
    cats2 = df[feat_s2].dropna().value_counts().head(3).index.tolist()
    resultados = ['Abandono', 'Éxito/Activo']

    nodos = [str(c) for c in cats1] + [str(c) for c in cats2] + resultados
    idx_map = {n: i for i, n in enumerate(nodos)}

    sources, targets, values, colors_link = [], [], [], []
    for c1 in cats1:
        for c2 in cats2:
            mask = (df[feat_s1]==c1) & (df[feat_s2]==c2)
            n = mask.sum()
            if n > 50:
                sources.append(idx_map[str(c1)])
                targets.append(idx_map[str(c2)])
                values.append(int(n))
                colors_link.append('rgba(160,174,192,0.4)')
    for c2 in cats2:
        for res_idx, res in enumerate(resultados):
            mask2 = (df[feat_s2]==c2) & (df[TARGET]==(1 if res_idx==0 else 0))
            n2 = mask2.sum()
            if n2 > 50:
                sources.append(idx_map[str(c2)])
                targets.append(idx_map[res])
                values.append(int(n2))
                colors_link.append('rgba(229,62,62,0.35)' if res_idx==0 else 'rgba(49,130,206,0.35)')

    fig_sank = go.Figure(go.Sankey(
        node=dict(
            pad=20, thickness=25,
            label=nodos,
            color=['#a0aec0']*len(cats1) + ['#4a5568']*len(cats2) +
                  [COLOR_ABND, COLOR_EXIT],
            hovertemplate='<b>%{label}</b><br>%{value} estudiantes<extra></extra>'
        ),
        link=dict(source=sources, target=targets, value=values, color=colors_link)
    ))
    fig_sank.update_layout(
        title=dict(
            text=f'Flujo de estudiantes: {feat_s1} → {feat_s2} → resultado',
            font=dict(size=13), x=0.5),
        height=560, paper_bgcolor='white',
        font=dict(size=13, color='#1a202c'),
        margin=dict(t=70, b=60, l=200, r=200)
    )
    IMG_SANKEY = fig_sank.to_html(full_html=False, include_plotlyjs='cdn')
    print('✅ Sankey OK')
else:
    IMG_SANKEY = '<p style="color:#718096;font-style:italic;">Sankey no disponible — variables categóricas insuficientes.</p>'
    print('⚠️  Sankey omitido')

# ── Bump chart ───────────────────────────────────────────────────────────────
df_bump = df_stats.head(10).copy().reset_index(drop=True)
df_bump['rank_diff'] = range(1, len(df_bump)+1)

if RUTA_RANKING.exists():
    df_rk = pd.read_parquet(RUTA_RANKING)
    df_rk.columns = [c.lower().strip() for c in df_rk.columns]
    col_rank = next((c for c in ['rango_medio','puntuacion_final','rango_m06','ranking']
                     if c in df_rk.columns), None)
    if col_rank and 'feature' in df_rk.columns:
        rk_map = df_rk.set_index('feature')[col_rank].to_dict()
        df_bump['val_m06'] = df_bump['feature'].map(
            lambda f: float(rk_map.get(
                f, df_bump.loc[df_bump['feature']==f,'rank_diff'].values[0])))
        df_bump['rank_m06'] = df_bump['val_m06'].rank(method='first').astype(int)
    else:
        df_bump['rank_m06'] = df_bump['rank_diff']
else:
    df_bump['rank_m06'] = df_bump['rank_diff']

fig_bump = go.Figure()

# Líneas de conexión
for _, row in df_bump.iterrows():
    rd, rm  = int(row['rank_diff']), int(row['rank_m06'])
    cambio  = rm - rd
    if cambio == 0:
        color = '#a0aec0'
    elif cambio > 0:
        color = COLOR_EXIT
    else:
        color = COLOR_ABND
    width = 1.5 + min(abs(cambio), 5) * 0.4
    fig_bump.add_trace(go.Scatter(
        x=[0, 1], y=[rd, rm],
        mode='lines',
        line=dict(color=color, width=width),
        showlegend=False,
        hovertemplate=(
            f'<b>{row["feature"]}</b><br>'
            f'Este módulo: #{rd} · M06: #{rm}<br>'
            f'Cambio: {rd-rm:+d} posiciones'
            '<extra></extra>'
        )
    ))

# Puntos izquierda
fig_bump.add_trace(go.Scatter(
    x=[0] * len(df_bump), y=df_bump['rank_diff'].tolist(),
    mode='markers+text',
    text=df_bump['feature'].str.replace('_',' ').tolist(),
    textposition='middle left', textfont=dict(size=9.5, color='#2d3748'),
    marker=dict(color=COLOR_ABND, size=13, line=dict(color='white', width=2)),
    name='Diferencia entre grupos', hoverinfo='skip'
))

# Puntos derecha
fig_bump.add_trace(go.Scatter(
    x=[1] * len(df_bump), y=df_bump['rank_m06'].tolist(),
    mode='markers+text',
    text=df_bump['feature'].str.replace('_',' ').tolist(),
    textposition='middle right', textfont=dict(size=9.5, color='#2d3748'),
    marker=dict(color=COLOR_EXIT, size=13, symbol='diamond',
                line=dict(color='white', width=2)),
    name='Ranking M06', hoverinfo='skip'
))

fig_bump.update_layout(
    title=dict(
        text='Bump chart — ranking por diferencia entre grupos vs ranking estadístico M06',
        font=dict(size=13), x=0.5),
    xaxis=dict(
        tickvals=[0, 1],
        ticktext=['<b>Este módulo</b><br>(diferencia entre grupos)',
                  '<b>Ranking M06</b><br>(correlación + info mutua)'],
        tickfont=dict(size=11, color='#2d3748'),
        showgrid=False, zeroline=False, range=[-0.55, 1.55]
    ),
    yaxis=dict(
        autorange='reversed',
        title='Posición (1 = más importante)',
        tickmode='linear', dtick=1,
        showgrid=True, gridcolor='#e2e8f0',
        range=[0.3, len(df_bump) + 0.7]
    ),
    legend=dict(
        orientation='h', yanchor='bottom', y=1.03,
        xanchor='center', x=0.5, font=dict(size=11),
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='#e2e8f0', borderwidth=1
    ),
    height=560,
    paper_bgcolor='white', plot_bgcolor='white',
    margin=dict(t=70, b=60, l=185, r=185)
)
IMG_BUMP = fig_bump.to_html(full_html=False, include_plotlyjs='cdn')
print('✅ Bump chart OK')

print('\n✅ Todos los gráficos de ampliación generados')


✅ Heatmaps OK


✅ Ridgeline OK
✅ Sankey OK
✅ Bump chart OK

✅ Todos los gráficos de ampliación generados


In [7]:
# ============================================================================
# CELDA 7: HTML PRINCIPAL — con botones 📖 Interpretar en todos los gráficos
# ============================================================================

kpis_html = (
    '<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));'
    'gap:16px;margin-bottom:28px;">'
    f'<div style="background:#fff5f5;border-left:4px solid {COLOR_ABND};'
    'padding:16px 20px;border-radius:8px;">'
    '<div style="font-size:11px;color:#4a5568;text-transform:uppercase;'
    'letter-spacing:.6px;margin-bottom:4px;">Abandonadores</div>'
    f'<div style="font-size:28px;font-weight:700;">{formato_numero_es(n_abnd)}</div>'
    f'<div style="font-size:12px;color:#4a5568;">{n_abnd/len(df)*100:.1f}% del dataset</div></div>'
    f'<div style="background:#ebf8ff;border-left:4px solid {COLOR_EXIT};'
    'padding:16px 20px;border-radius:8px;">'
    '<div style="font-size:11px;color:#4a5568;text-transform:uppercase;'
    'letter-spacing:.6px;margin-bottom:4px;">Éxito / Activos</div>'
    f'<div style="font-size:28px;font-weight:700;">{formato_numero_es(n_exit)}</div>'
    f'<div style="font-size:12px;color:#4a5568;">{n_exit/len(df)*100:.1f}% del dataset</div></div>'
    '<div style="background:#f0fff4;border-left:4px solid #48bb78;'
    'padding:16px 20px;border-radius:8px;">'
    '<div style="font-size:11px;color:#4a5568;text-transform:uppercase;'
    'letter-spacing:.6px;margin-bottom:4px;">Features analizadas</div>'
    f'<div style="font-size:28px;font-weight:700;">{len(df_stats)}</div></div>'
    '<div style="background:#fffbeb;border-left:4px solid #d69e2e;'
    'padding:16px 20px;border-radius:8px;">'
    '<div style="font-size:11px;color:#4a5568;text-transform:uppercase;'
    'letter-spacing:.6px;margin-bottom:4px;">Mayor diferencia</div>'
    f'<div style="font-size:16px;font-weight:700;">{df_stats.iloc[0]["feature"]}</div>'
    f'<div style="font-size:12px;color:#4a5568;">'
    f'abandono={df_stats.iloc[0]["med_abnd"]:.2f} · éxito={df_stats.iloc[0]["med_exit"]:.2f}'
    '</div></div>'
    '</div>'
)

js_css_html = (
    '<style>'
    '.ibt{display:inline-flex;align-items:center;gap:4px;background:#3182ce;'
    'color:#fff;border:none;border-radius:6px;padding:4px 12px;font-size:11px;'
    'font-weight:600;cursor:pointer;transition:background .2s;white-space:nowrap;}'
    '.ibt:hover{background:#2b6cb0;}'
    '.ipn{display:none;background:#ebf8ff;border:1px solid #bee3f8;'
    'border-radius:8px;padding:14px 18px;margin:6px 0 18px;font-size:13px;'
    'color:#2d3748;line-height:1.75;}'
    '.ipn.vis{display:block;}'
    '.ipn ul{margin:6px 0 0;padding-left:18px;}'
    '.ipn li{margin-bottom:5px;}'
    '.tg{color:#276749;font-weight:600;}'
    '.tm{color:#744210;font-weight:600;}'
    '.tb{color:#9b2c2c;font-weight:600;}'
    '</style>'
    '<script>'
    'function tg(id){'
    'var e=document.getElementById(id);'
    'var btn=e.previousElementSibling.querySelector(".ibt");'
    'e.classList.toggle("vis");'
    'btn.textContent=e.classList.contains("vis")?"✖ Cerrar":"📖 Interpretar";}'
    '</script>'
)

def h2btn(titulo, pid):
    return (
        '<div style="display:flex;align-items:center;gap:12px;margin:32px 0 4px;">'
        f'<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:0;flex:1;">'
        f'{titulo}</h2>'
        f'<button class="ibt" onclick="tg(\'{pid}\')">📖 Interpretar</button>'
        '</div>'
    )

def panel(pid, contenido):
    return f'<div id="{pid}" class="ipn">{contenido}</div>'

# Paneles con interpretaciones dinámicas calculadas en celda 5
p_radar = panel('p-radar', radar_interp_html)
p_violin = panel('p-violin', violin_interp_html)
p_dot = panel('p-dot', dot_interp_html)
p_kde = panel('p-kde', kde_interp_html)
p_cat = panel('p-cat', cat_interp_html)
p_comp = panel('p-comp', comp_interp_html)
p_sem = panel('p-sem', sem_interp_html)

top_feat   = df_stats.iloc[0]['feature']
top_diff   = abs(df_stats.iloc[0]['diff_abs'])
prot_row   = df_stats.loc[df_stats['diff_abs'].idxmin()]
riesgo_row = df_stats.loc[df_stats['diff_abs'].idxmax()]

bloque_conclusiones = (
    '<div style="background:#fffbeb;border:1px solid #f6e05e;border-radius:8px;'
    'padding:20px;margin-top:32px;">'
    '<h3 style="color:#744210;font-size:15px;margin:0 0 14px;">📋 Conclusiones</h3>'
    f'<p style="color:#4a5568;font-size:14px;margin:0 0 10px;">'
    f'<strong>🔍 Mayor diferencia:</strong> <strong>{top_feat}</strong>'
    f' — diferencia de medianas = {top_diff:.2f}.</p>'
    f'<p style="color:#4a5568;font-size:14px;margin:0 0 10px;">'
    f'<strong>⚠️ Factor de riesgo principal:</strong> <strong>{riesgo_row["feature"]}</strong>'
    f' — abandono={riesgo_row["med_abnd"]:.2f}, éxito={riesgo_row["med_exit"]:.2f}.</p>'
    f'<p style="color:#4a5568;font-size:14px;margin:0 0 10px;">'
    f'<strong>🛡️ Factor protector principal:</strong> <strong>{prot_row["feature"]}</strong>'
    f' — éxito={prot_row["med_exit"]:.2f} vs abandono={prot_row["med_abnd"]:.2f}.</p>'
    f'<p style="color:#4a5568;font-size:14px;margin:0;">'
    f'<strong>🔭 Línea futura:</strong> el dataset permite desarrollar un modelo de '
    f'<strong>regresión</strong> para predecir la nota media esperada de los '
    f'{formato_numero_es(n_exit)} estudiantes que completan la carrera '
    f'({n_exit/len(df)*100:.1f}% del dataset). '
    'Identificado como trabajo futuro prioritario.</p>'
    '</div>'
)

boton_ampliacion = (
    '<div style="text-align:center;margin:32px 0;">'
    '<a href="m08_perfiles_riesgo_ampliacion.html" '
    'style="display:inline-block;background:#3182ce;color:#fff;'
    'padding:12px 28px;border-radius:8px;font-weight:600;font-size:14px;'
    'text-decoration:none;">'
    '🔬 Ver análisis ampliado →</a>'
    '<p style="color:#718096;font-size:12px;margin:8px 0 0;">'
    'Heatmap · Ridgeline · Sankey · Bump chart</p>'
    '</div>'
)

cuerpo_principal = (
    js_css_html
    + kpis_html
    + h2btn('Perfil de riesgo — radar', 'p-radar')
    + p_radar
    + f'<p style="color:#4a5568;font-size:13px;margin-bottom:12px;">'
    f'Top {len(feats_radar)} features normalizadas (0–1). '
    f'<span style="color:{COLOR_ABND};font-weight:600;">Rojo = abandono</span> · '
    f'<span style="color:{COLOR_EXIT};font-weight:600;">Azul = éxito</span>.</p>'
    + IMG_RADAR
    + h2btn('Distribución por grupo — violín dividido', 'p-violin')
    + p_violin
    + '<p style="color:#4a5568;font-size:13px;margin-bottom:12px;">'
    'Izquierda = abandonador · Derecha = éxito. Escala P1–P99 por feature.</p>'
    + IMG_VIOLIN
    + h2btn('Medias con intervalo de confianza al 95%', 'p-dot')
    + p_dot
    + '<p style="color:#4a5568;font-size:13px;margin-bottom:12px;">'
    'Media normalizada (0–1) con IC 95%. Barras sin tocarse = diferencia significativa.</p>'
    + IMG_DOT_IC
    + h2btn('Densidad de kernel — KDE', 'p-kde')
    + p_kde
    + '<p style="color:#4a5568;font-size:13px;margin-bottom:12px;">'
    'Curvas de densidad suavizadas. Separadas = buen predictor · solapadas = discrimina poco.</p>'
    + IMG_KDE
    + h2btn('Variables categóricas — % por grupo', 'p-cat')
    + p_cat
    + '<p style="color:#4a5568;font-size:13px;margin-bottom:12px;">'
    'Porcentaje de abandono vs éxito por cada categoría.</p>'
    + IMG_CAT
    + h2btn('Comparativa de medianas', 'p-comp')
    + p_comp
    + '<p style="color:#4a5568;font-size:13px;margin-bottom:12px;">'
    'Mediana real con rango P25–P75 como barras de error.</p>'
    + IMG_COMP
    + h2btn('Tabla semáforo', 'p-sem')
    + p_sem
    + '<p style="color:#4a5568;font-size:13px;margin-bottom:12px;">'
    '🟢 Protector · 🔴 Factor de riesgo · 🟡 Solapamiento.</p>'
    + tabla_semaforo_html
    + bloque_conclusiones
    + boton_ampliacion
)

nav_fases, nav_modulos = generar_html_navegacion_completa(
    fase_activa='fase4', modulo_activo='m08')
html_principal = render_base_html(
    titulo=TITULO_PAGINA, icono='📊', subtitulo=SUBTITULO,
    nav_fases=nav_fases, nav_modulos=nav_modulos,
    contenido=cuerpo_principal,
    notebook_nombre='f4_m08_perfiles_riesgo.ipynb',
    notebook_carpeta='fase4_eda'
)
ruta_principal = RUTA_FASE4_HTML / 'm08_perfiles_riesgo.html'
guardar_html(html_principal, ruta_principal)
print(f'✅ HTML principal: {ruta_principal}')


✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m08_perfiles_riesgo.html


✅ HTML principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m08_perfiles_riesgo.html


In [8]:
# ============================================================================
# CELDA 8: HTML AMPLIACIÓN
# ============================================================================

boton_volver = (
    '<div style="margin-bottom:24px;">'
    '<a href="m08_perfiles_riesgo.html" '
    'style="display:inline-block;background:#edf2f7;color:#2d3748;'
    'padding:8px 20px;border-radius:6px;font-size:13px;'
    'text-decoration:none;font-weight:600;">'
    '← Volver al análisis principal</a>'
    '</div>'
)

cuerpo_ampliacion = (
    js_css_html
    + boton_volver
    + h2btn('Heatmap de correlaciones por grupo', 'p-heat')
    + panel('p-heat',
        '<b>¿Qué muestra?</b> La correlación lineal entre features dentro de cada grupo por separado.<br>'
        'Si dos features tienen correlación alta en abandono pero no en éxito = '
        'esa relación es específica del grupo, información valiosa para SHAP en Fase 6.<br>'
        '<b>Colores:</b> azul oscuro = correlación positiva fuerte · rojo = negativa · blanco = sin correlación.'
    )
    + '<p style="color:#4a5568;font-size:13px;margin-bottom:12px;">'
    'Correlaciones internas de cada grupo. Si difieren entre grupos = señal para SHAP.</p>'
    + '<div style="display:flex;gap:16px;flex-wrap:wrap;">'
    + f'<div style="flex:1;min-width:400px;">{IMG_HEAT_ABND}</div>'
    + f'<div style="flex:1;min-width:400px;">{IMG_HEAT_EXIT}</div>'
    + '</div>'
    + h2btn('Ridgeline plot — distribución apilada por feature', 'p-ridge')
    + panel('p-ridge', ridge_interp_html)
    + '<p style="color:#4a5568;font-size:13px;margin-bottom:12px;">'
    'Cada fila es una feature normalizada (0–1). '
    'Curvas separadas = buen predictor · solapadas = discrimina poco.</p>'
    + IMG_RIDGE
    + h2btn('Diagrama Sankey — flujo de estudiantes', 'p-sank')
    + panel('p-sank',
        '<b>¿Qué muestra?</b> El flujo de estudiantes a través de dos variables categóricas hasta el resultado.<br>'
        '<b>Nodos grises izquierda</b> = categorías de vía de acceso. '
        '<b>Nodos oscuros centro</b> = situación laboral. '
        '<b>Grosor de banda</b> = número de estudiantes que recorren ese camino.<br>'
        f'<span style="color:{COLOR_ABND};font-weight:600;">Bandas rojas</span> = fluyen a abandono · '
        f'<span style="color:{COLOR_EXIT};font-weight:600;">Bandas azules</span> = fluyen a éxito.'
    )
    + '<p style="color:#4a5568;font-size:13px;margin-bottom:12px;">'
    f'Flujo de estudiantes: vía de acceso → situación laboral → resultado final. '
    f'<span style="color:{COLOR_ABND};font-weight:600;">Rojo = abandono</span> · '
    f'<span style="color:{COLOR_EXIT};font-weight:600;">Azul = éxito</span>.</p>'
    + IMG_SANKEY
    + '<div style="background:#f7fafc;border:1px solid #e2e8f0;border-radius:6px;'
    'padding:12px 16px;margin-top:8px;font-size:12px;color:#4a5568;">'
    f'📖 Los nodos <b>grises</b> de la izquierda son las categorías de vía de acceso. '
    f'Los nodos <b>oscuros</b> del centro son situación laboral. '
    f'<span style="color:{COLOR_ABND};font-weight:600;">Rojo</span> = fluye a abandono · '
    f'<span style="color:{COLOR_EXIT};font-weight:600;">Azul</span> = fluye a éxito. '
    'El grosor es proporcional al número de estudiantes.'
    + '</div>'
    + h2btn('Bump chart — coherencia entre rankings', 'p-bump')
    + panel('p-bump',
        '<b>¿Qué muestra?</b> Cada línea conecta la misma feature en dos rankings diferentes.<br>'
        '<b>Izquierda (rojo)</b> = ranking según diferencia de medianas entre grupos (este módulo).<br>'
        '<b>Derecha (azul)</b> = ranking estadístico de M06 (correlación + información mutua).<br>'
        '<b>Línea horizontal</b> = ambas fuentes coinciden en la importancia de esa feature.<br>'
        '<b>Línea que cruza</b> = discrepancia: merece análisis con SHAP en Fase 6 para resolver cuál tiene razón.'
    )
    + '<p style="color:#4a5568;font-size:13px;margin-bottom:12px;">'
    '¿Coincide la importancia por diferencia de grupos con el ranking estadístico M06?</p>'
    + IMG_BUMP
    + '<div style="background:#f7fafc;border:1px solid #e2e8f0;border-radius:6px;'
    'padding:12px 16px;margin-top:8px;font-size:12px;color:#4a5568;">'
    f'📖 <b>Cómo leer:</b> '
    f'<span style="color:{COLOR_ABND};font-weight:600;">Línea roja</span> = baja posiciones en M06 · '
    f'<span style="color:{COLOR_EXIT};font-weight:600;">Línea azul</span> = sube en M06 · '
    '<span style="color:#a0aec0;font-weight:600;">Gris</span> = sin cambio. '
    'El grosor es proporcional a la magnitud del cambio.'
    + '</div>'
    + boton_volver
)

nav_fases, nav_modulos = generar_html_navegacion_completa(
    fase_activa='fase4', modulo_activo='m08')
html_ampliacion = render_base_html(
    titulo=TITULO_PAGINA + ' — Análisis Ampliado',
    icono='🔬', subtitulo=SUBTITULO,
    nav_fases=nav_fases, nav_modulos=nav_modulos,
    contenido=cuerpo_ampliacion,
    notebook_nombre='f4_m08_perfiles_riesgo.ipynb',
    notebook_carpeta='fase4_eda'
)
ruta_ampliacion = RUTA_FASE4_HTML / 'm08_perfiles_riesgo_ampliacion.html'
guardar_html(html_ampliacion, ruta_ampliacion)
print(f'✅ HTML ampliación: {ruta_ampliacion}')
print()
print('=' * 55)
print('COMPLETADO: F4-M08')
print(f'  Principal : {RUTA_FASE4_HTML}/m08_perfiles_riesgo.html')
print(f'  Ampliación: {RUTA_FASE4_HTML}/m08_perfiles_riesgo_ampliacion.html')
print('=' * 55)


✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m08_perfiles_riesgo_ampliacion.html
✅ HTML ampliación: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m08_perfiles_riesgo_ampliacion.html

COMPLETADO: F4-M08
  Principal : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4/m08_perfiles_riesgo.html
  Ampliación: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4/m08_perfiles_riesgo_ampliacion.html
